# FreightQuote AI — Milestone 4: Maritime Knowledge RAG Builder
This notebook builds/updates the maritime knowledge base and generates the retrieval indexes (FAISS dense + BM25 sparse) required by the Streamlit application.

### Step 1: Install Required Dependencies
Installs the packages required for PDF text extraction, character splitting, FAISS database indexing, and BM25 search.

In [ ]:
!pip install pymupdf reportlab langchain langchain-community langchain-text-splitters rank_bm25 sentence-transformers faiss-cpu

### Step 2: Establish Working Directories
Creates directories under `/content/freightquote_m4` to keep models, document sources, and indexes organized.

In [ ]:
import os
BASE_DIR = "/content/freightquote_m4"
RAG_DIR = os.path.join(BASE_DIR, "rag_documents")
FAISS_DIR = os.path.join(BASE_DIR, "faiss_index")
MODELS_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(RAG_DIR, exist_ok=True)
os.makedirs(FAISS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Directories established under: {BASE_DIR}")

### Step 3: Google Drive & Document Upload Portal
Mount Google Drive (optional) or upload local files.

In [ ]:
from google.colab import files, drive
import glob

st_drive = input("Would you like to mount Google Drive? (y/n): ").strip().lower()
if st_drive == 'y':
    drive.mount('/content/drive')
    drive_folder = input("Enter path to your Drive documents folder (e.g. /content/drive/MyDrive/maritime_docs): ").strip()
    if os.path.exists(drive_folder):
        import shutil
        for f in glob.glob(os.path.join(drive_folder, "*.pdf")):
            shutil.copy(f, RAG_DIR)
        print(f"Copied PDFs from Google Drive to {RAG_DIR}")
    else:
        print("Drive directory path not found.")

st_upload = input("Would you like to upload local PDF files? (y/n): ").strip().lower()
if st_upload == 'y':
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(RAG_DIR, name), "wb") as f:
            f.write(content)
    print(f"Uploaded {len(uploaded)} documents successfully.")

### Step 4: Write Document Generator Utility
Writes the ReportLab document generator containing the standard 35 seed documents in case the knowledge folder is empty.

In [ ]:
%%writefile /content/freightquote_m4/vector_store.py
"""
vector_store.py — FAISS Vector Database Management Module for FreightQuote AI.
Handles PDF document generation, text extraction, semantic chunking, and FAISS indexing.
"""
import os
import glob
import fitz  # PyMuPDF
from typing import List
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    try:
        from langchain.text_splitter import RecursiveCharacterTextSplitter
    except ImportError:
        RecursiveCharacterTextSplitter = None

try:
    from langchain_core.documents import Document
except ImportError:
    try:
        from langchain.docstore.document import Document
    except ImportError:
        Document = None

try:
    from langchain_community.vectorstores import FAISS
except ImportError:
    try:
        from langchain.vectorstores import FAISS
    except ImportError:
        FAISS = None

import config
from embeddings import get_embedding_model

_vectorstore = None

def generate_pdf(filename, title, category, intro, sections, table_data, bullets):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    doc = SimpleDocTemplate(filename, pagesize=letter,
                            rightMargin=54, leftMargin=54,
                            topMargin=54, bottomMargin=54)
    styles = getSampleStyleSheet()
    
    title_style = ParagraphStyle(
        'DocTitle',
        parent=styles['Heading1'],
        fontName='Helvetica-Bold',
        fontSize=20,
        leading=24,
        textColor=colors.HexColor('#1A365D'),
        spaceAfter=15
    )
    
    cat_style = ParagraphStyle(
        'DocCat',
        parent=styles['Normal'],
        fontName='Helvetica-Bold',
        fontSize=9,
        leading=11,
        textColor=colors.HexColor('#D69E2E'),
        spaceAfter=20
    )
    
    h2_style = ParagraphStyle(
        'DocH2',
        parent=styles['Heading2'],
        fontName='Helvetica-Bold',
        fontSize=12,
        leading=15,
        textColor=colors.HexColor('#2C5282'),
        spaceBefore=14,
        spaceAfter=8
    )
    
    body_style = ParagraphStyle(
        'DocBody',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=10,
        leading=14,
        textColor=colors.HexColor('#2D3748'),
        spaceAfter=10
    )
    
    bullet_style = ParagraphStyle(
        'DocBullet',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=10,
        leading=14,
        textColor=colors.HexColor('#2D3748'),
        leftIndent=20,
        firstLineIndent=-10,
        spaceAfter=6
    )
    
    story = []
    
    # Title & Category
    story.append(Paragraph(title, title_style))
    story.append(Paragraph(f"FREIGHTQUOTE AI KNOWLEDGE SYSTEM &mdash; CATEGORY: {category.upper()}", cat_style))
    story.append(Spacer(1, 10))
    
    # Intro
    story.append(Paragraph("<b>1. EXECUTIVE SUMMARY & OVERVIEW</b>", h2_style))
    story.append(Paragraph(intro, body_style))
    story.append(Spacer(1, 10))
    
    # Custom Sections
    for idx, (sec_title, sec_text) in enumerate(sections, 2):
        story.append(Paragraph(f"<b>{idx}. {sec_title.upper()}</b>", h2_style))
        for p in sec_text:
            story.append(Paragraph(p, body_style))
        story.append(Spacer(1, 10))
        
    story.append(PageBreak()) # Force to Page 2 to ensure multi-page requirement
    
    # Table of standards
    story.append(Paragraph(f"<b>{len(sections)+2}. PROCESS METRICS & COMPLIANCE TARGETS</b>", h2_style))
    story.append(Spacer(1, 8))
    
    table_content = []
    # Header
    table_content.append([
        Paragraph("<b>Operation/Component</b>", body_style),
        Paragraph("<b>Standard Parameter</b>", body_style),
        Paragraph("<b>KPI Target</b>", body_style),
        Paragraph("<b>Mitigation Rule</b>", body_style)
    ])
    for row in table_data:
        table_content.append([
            Paragraph(row[0], body_style),
            Paragraph(row[1], body_style),
            Paragraph(row[2], body_style),
            Paragraph(row[3], body_style)
        ])
    
    t = Table(table_content, colWidths=[120, 140, 100, 140])
    t.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#E2E8F0')),
        ('ALIGN', (0,0), (-1,-1), 'LEFT'),
        ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ('BOTTOMPADDING', (0,0), (-1,0), 6),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CBD5E0')),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F7FAFC')]),
        ('TOPPADDING', (0,0), (-1,-1), 6),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    ]))
    story.append(t)
    story.append(Spacer(1, 15))
    
    # Bullet points
    story.append(Paragraph(f"<b>{len(sections)+3}. STRATEGIC RECOMMENDATIONS & KEY TAKEAWAYS</b>", h2_style))
    for b in bullets:
        story.append(Paragraph(f"&bull; {b}", bullet_style))
        
    doc.build(story)

# Corpus data representing 38 documents (19 original + 19 new)

corpus_data = [
    {
        "filename": "01_Freight_Rates_Calculations.pdf",
        "title": "Freight Costs and Rate Structures",
        "category": "Freight",
        "intro": "Freight rates form the core pricing structure of global commerce. They represent the charge for transporting goods from origin to destination and depend on cargo type, transport mode, and handling requirements. This document outlines volumetric calculations, ocean shipping surcharges, demurrage, and detention guidelines.",
        "sections": [
            ("Chargeable Weight Calculations", [
                "Chargeable weight is calculated based on the greater of Actual (Gross) Weight and Volumetric (Dimensional) Weight. Volumetric weight is calculated using the cargo volume in cubic meters (CBM) multiplied by a modal factor.",
                "For Air Freight, the standard ratio is 1 CBM = 167 kilograms (1:6 scale). Thus, Volumetric Weight (kg) = Length (m) x Width (m) x Height (m) x 167.",
                "For Ocean Freight, the ratio is 1 CBM = 1,000 kilograms (1:1 scale). For Road Freight (LTL), it varies by region, but standard European trucking uses 1 CBM = 333 kilograms (1:3 scale).",
            ]),
            ("Standard Ocean Surcharges", [
                "Bunker Adjustment Factor (BAF): A fluctuating surcharge added to ocean freight rates to compensate for price volatility in marine fuel.",
                "Currency Adjustment Factor (CAF): A surcharge added to offset exchange rate fluctuations between the billing currency (often USD) and carrier cost currencies.",
                "Peak Season Surcharge (PSS): A temporary surcharge applied by ocean carriers during peak shipping months (typically August through October) in anticipation of increased holiday retail volumes.",
            ]),
            ("Demurrage and Detention Policies", [
                "Demurrage: Surcharges applied by port terminals when containers remain inside the port facility beyond the allowed free days (typically 3 to 7 days).",
                "Detention: Surcharges applied by carriers when containers are removed from the port but not returned empty to the carrier's container yard within the free time limit.",
            ])
        ],
        "table_data": [
            ["Air Freight Ratio", "1 CBM = 167 kg", "Strict 1:6 ratio", "Verify dimensional weight before quote"],
            ["Ocean Freight Ratio", "1 CBM = 1,000 kg", "Strict 1:1 ratio", "Optimize container packing space"],
            ["Demurrage Free Time", "3 - 7 days standard", "100% on-time return", "Pre-clear customs to avoid port dwell"],
            ["Detention Free Time", "5 - 10 days standard", "Zero carrier penalties", "Coordinate empty return with drayage"]
        ],
        "bullets": [
            "Always compare volumetric weight with gross weight before preparing shipment documentation.",
            "Verify carrier-specific BAF and CAF indices weekly as fuel and currency markets fluctuate.",
            "Maintain pre-negotiated demurrage waivers (up to 14 days) for high-volume trade routes.",
            "Utilize digital port dwell alerts to prompt import drayage dispatch immediately upon vessel discharge.",
            "Consolidate cargo to minimize LCL minimum-weight charges on low-density freight."
        ]
    },
    {
        "filename": "02_Incoterms_2020_Guide.pdf",
        "title": "Incoterms 2020 Operational Standards",
        "category": "Incoterms",
        "intro": "Incoterms (International Commercial Terms) are standardized rules published by the International Chamber of Commerce (ICC). They define the risk transfer, insurance duties, and freight cost allocation between buyers and sellers in international trade transactions.",
        "sections": [
            ("EXW (Ex Works) and FOB (Free On Board)", [
                "EXW (Ex Works) represents the minimum obligation for the seller. The seller makes the goods available at their premises. The buyer bears all costs and risks from the seller's facility.",
                "FOB (Free On Board) requires the seller to clear the goods for export and deliver them onboard the vessel designated by the buyer at the named port of shipment. Risk transfers to the buyer when the goods are onboard.",
            ]),
            ("CIF (Cost, Insurance & Freight) and DDP (Delivered Duty Paid)", [
                "CIF requires the seller to pay the costs and freight to bring the goods to the named destination port, and to purchase minimum marine insurance coverage. Risk transfers to the buyer once the goods are onboard at the origin port.",
                "DDP represents the maximum obligation for the seller. The seller is responsible for delivering the goods to the buyer's destination, cleared for import, and paying all customs duties, taxes, and terminal charges.",
            ])
        ],
        "table_data": [
            ["EXW", "Seller factory floor", "Buyer assumes 100% risk", "Buyer arranges freight and customs"],
            ["FOB", "Origin port vessel deck", "Split risk at rail", "Seller handles export, buyer handles sea"],
            ["CIF", "Destination port vessel", "Seller pays freight/insurance", "Risk transfers at origin loading"],
            ["DDP", "Buyer named warehouse", "Seller assumes 100% risk", "Seller pays import duties and taxes"]
        ],
        "bullets": [
            "Ensure purchase contracts explicitly state the named place alongside the Incoterms version (e.g., FOB Shanghai Incoterms 2020).",
            "Avoid EXW for international shipments unless the buyer has a robust local presence to manage export compliance.",
            "Use DDP only when the seller has established importing capabilities and tax registration in the destination country.",
            "Establish cargo insurance levels exceeding CIF minimums (which only cover 110% of invoice value).",
            "Audit freight forwarder invoices to ensure cost allocations align with the contracted Incoterm."
        ]
    },
    {
        "filename": "03_Customs_Clearance_Compliance.pdf",
        "title": "Customs Clearance and Documentation Guidelines",
        "category": "Customs",
        "intro": "Customs clearance is a regulatory process required for goods entering or leaving a country. Compliance with local laws, correct documentation, and accurate tariff classifications are critical to prevent shipping delays and heavy financial penalties.",
        "sections": [
            ("Mandatory Shipping Documentation", [
                "Commercial Invoice: Detailing buyer, seller, transaction value, descriptions, country of origin, and Incoterms.",
                "Packing List: Itemizing net and gross weights, dimensions, container numbers, and package counts.",
                "Bill of Lading / Air Waybill: Serving as a contract of carriage and receipt of goods.",
                "Certificate of Origin: Validating where the goods were manufactured to determine tariff rates.",
            ]),
            ("Harmonized System (HS) Codes & Filing", [
                "HS Codes are 6-to-10 digit numbers used globally to classify products. The first 6 digits are standardized worldwide; additional digits are set by individual countries.",
                "Electronic filings are processed via national portals, such as ICEGATE in India and ACE (Automated Commercial Environment) in the United States.",
            ])
        ],
        "table_data": [
            ["Commercial Invoice", "Must show Transaction Value", "100% accurate description", "Errors lead to valuation holds"],
            ["HS Classification", "6-to-10 digit codes", "Correct tariff rate application", "Misclassification triggers fines"],
            ["Electronic Portals", "ICEGATE / ACE", "Pre-arrival manifest submission", "24-hour rule compliance mandatory"],
            ["Post-Clearance Audit", "Valuation / Origin checks", "5-year document retention", "Maintains corporate compliance status"]
        ],
        "bullets": [
            "Obtain Binding Tariff Information (BTI) from customs authorities for complex or multi-component items.",
            "Transmit import manifests at least 24 hours prior to origin port loading to comply with container security regulations.",
            "Reconcile commercial invoice values with actual bank transaction records to prove transaction value during audits.",
            "Verify bilateral Free Trade Agreement eligibility before declaring origin on customs entry forms.",
            "Retain all customs entry documents (e.g., Bill of Entry, customs receipt) for a minimum of 5 years."
        ]
    },
    {
        "filename": "04_Port_Congestion_Dwell_Times.pdf",
        "title": "Port Logistics and Container Dwell Times",
        "category": "Port",
        "intro": "Container dwell time represents the time a container spends at a port terminal after being discharged from a vessel or before being loaded onto one. High dwell times are key indicators of supply chain bottlenecks and port congestion.",
        "sections": [
            ("Drivers of Container Dwell Times", [
                "Inland transport shortages (truck driver scarcities, rail car unavailability) block container evacuation.",
                "Customs documentation delays and manual clearance processes stall container release from terminal yards.",
                "Vessel bunching and high peak volume arrivals overwhelm port handling equipment and staff.",
            ]),
            ("Mitigation Technologies & Strategies", [
                "Automated Rubber-Tired Gantry Cranes (RTGC) and Automated Guided Vehicles (AGVs) optimize yard stacking and speed up truck turn times.",
                "Pre-gate appointment scheduling systems distribute truck arrivals evenly across terminal working hours.",
            ])
        ],
        "table_data": [
            ["Yard Crane Automation", "Automated RTGCs", "Reduce container retrieval by 35%", "Reduces yard dwell bottlenecks"],
            ["Pre-Gate Systems", "Truck Appointment systems", "Reduce truck turn times to <45 mins", "Maintains gate throughput"],
            ["Customs Integration", "Automated release links", "Instant port exit authorization", "Evacuates containers within 24 hours"],
            ["Inland Drayage", "Dedicated chassis pools", "Prevent transport delays", "Reduces port storage penalties"]
        ],
        "bullets": [
            "Monitor vessel schedule reliability and blank sailing announcements weekly to anticipate terminal yard stacking spikes.",
            "Utilize off-peak gate hours (night gates) to bypass peak truck traffic and reduce drayage costs.",
            "Deploy GPS-enabled smart container trackers to monitor real-time location inside the yard.",
            "Establish drayage partnerships that utilize dedicated chassis pools at congested ports.",
            "Divert container flows to secondary regional ports when primary gateways experience high vessel dwell."
        ]
    },
    {
        "filename": "05_Supply_Chain_Risk_Management.pdf",
        "title": "Supply Chain Risk and Disruptions Management",
        "category": "Risk",
        "intro": "Modern supply chains face threats from natural disasters, geopolitical tensions, labor disputes, and transport infrastructure failures. Building resilience requires systematic risk identification, scoring, and proactive mitigation.",
        "sections": [
            ("Types of Disruptions & Bottlenecks", [
                "Geopolitical Risks: Trade disputes, embargoes, tariff wars, and maritime security threats (e.g., piracy).",
                "Infrastructure Bottlenecks: Canal blockages (e.g., Suez or Panama Canal disruptions) and key transport hubs closure.",
                "Supplier Failure: Insolvency, production quality shutdowns, and financial distress of sole-source suppliers.",
            ]),
            ("Mitigation Strategies & Resilience", [
                "Dual-sourcing and multi-sourcing: Establishing secondary suppliers in different geographic regions.",
                "Strategic safety stocks: Maintaining buffer inventory for high-risk components.",
                "Real-time visibility: Integrating Automatic Identification System (AIS) tracking and supply chain dashboards.",
            ])
        ],
        "table_data": [
            ["Dual Sourcing", "Secondary supplier active", "Reduce supply loss risk by 50%", "Prevents single-point shutdowns"],
            ["Safety Stock", "Calculated buffer levels", "99% service level guarantee", "Covers transit delay variability"],
            ["Vessel Tracking", "AIS tracking updates", "Real-time ETA adjustments", "Enables proactive rerouting protocols"],
            ["General Average", "Marine insurance clause", "Covers cargo rescue contribution", "Secures release of detained cargo"]
        ],
        "bullets": [
            "Conduct quarterly vulnerability assessments for all tier-1 and critical tier-2 suppliers.",
            "Structure transit insurance policies to include comprehensive cargo damage and delay clauses.",
            "Incorporate General Average clauses in marine insurance to cover emergency cargo sacrifice expenses.",
            "Establish alternative logistics route options (e.g., sea-air, rail-sea) before crisis events occur.",
            "Maintain digital business continuity manuals with clear escalation rules for logistics managers."
        ]
    },
    {
        "filename": "06_Cold_Chain_Warehousing.pdf",
        "title": "Cold Chain and Temperature Controlled Warehousing",
        "category": "Warehousing",
        "intro": "Cold chain warehousing is a specialized logistics segment focused on storing temperature-sensitive products, including food, pharmaceuticals, and vaccines, under controlled environmental conditions.",
        "sections": [
            ("Temperature Controls & Compliance", [
                "Ultra-Cold (-70°C or lower) for specific vaccines, Frozen (-20°C) for meats and pharmaceuticals, and Chilled (2°C to 8°C) for fresh produce and biologics.",
                "Reefer containers equipped with active diesel-electric generator units (gensets) maintain continuous temperature monitoring during transit.",
                "FDA and WHO regulations mandate strict temperature records, validation audits, and environmental maps.",
            ]),
            ("Smart Cold Chain Warehouses", [
                "Automated Storage and Retrieval Systems (ASRS) optimize storage density and minimize human presence in sub-zero environments.",
                "IoT-enabled wireless sensors transmit real-time temperature and humidity logs to central ERP systems.",
            ])
        ],
        "table_data": [
            ["Chilled Zone", "2°C to 8°C range", "Zero thermal excursions", "Critical for vaccines and biologics"],
            ["Reefer Containers", "Active genset monitoring", "Continuous telemetry logs", "Maintains cargo integrity at sea"],
            ["ASRS Integration", "Robotic warehouse racking", "Minimize open-door heat loss", "Saves energy and improves safety"],
            ["Data Loggers", "Calibrated USB/BLE sensors", "Regulatory-grade reports", "Essential for FDA import compliance"]
        ],
        "bullets": [
            "Equip all cold chain shipments with redundant, independent temperature data loggers.",
            "Perform semi-annual temperature mapping audits inside warehouses to locate hot spots.",
            "Establish clear Standard Operating Procedures (SOPs) for transferring goods from cold storage to trucks.",
            "Verify forwarder capabilities in managing cold-chain transshipments at ocean hub ports.",
            "Install backup generators with automatic transfer switches to maintain power during outages."
        ]
    },
    {
        "filename": "07_Carrier_Selection_Auditing.pdf",
        "title": "Carrier Performance and Auditing Standards",
        "category": "Carrier Selection",
        "intro": "Carrier selection and auditing is a critical procurement process to ensure that logistics partners deliver on-time service, maintain compliance, and optimize transport spend.",
        "sections": [
            ("Key Performance Indicators (KPIs)", [
                "On-Time In-Full (OTIF): Measuring the percentage of shipments delivered within the scheduled window and containing the correct quantities.",
                "Damage Claim Ratio: Assessing carrier handling quality; target ratio is typically under 0.1% of shipments.",
                "Billing Accuracy: Auditing carrier invoice matching against contracted rates and fuel tables.",
            ]),
            ("Compliance and Safety Audits", [
                "Verifying carrier Certificate of Insurance (COI) liability limits and policy renewal dates.",
                "Monitoring safety ratings from regulatory bodies (such as FMCSA safety scores in the US).",
            ])
        ],
        "table_data": [
            ["OTIF Target", ">= 95% delivery rate", "Standard freight metric", "Penalties applied below threshold"],
            ["Damage Claims", "< 0.1% claim rate", "Maintains customer satisfaction", "Recurring claims trigger audits"],
            ["COI Verification", "Active liability limits", "100% policy compliance", "Restricts uncertified trucks"],
            ["Tariff Audits", "Invoice rating checks", "Zero double-billing errors", "Recovers overpayments annually"]
        ],
        "bullets": [
            "Use automated freight auditing software to check carrier invoice ratings against digital tariff tables.",
            "Establish a tier-based carrier rating system (e.g., Tier-1, Tier-2) to allocate freight volumes dynamically.",
            "Review carrier safety scores and compliance logs monthly before dispatching hazmat cargo.",
            "Conduct quarterly business reviews (QBR) with top-spend carriers to review performance metrics.",
            "Maintain secondary and backup carrier contracts to safeguard freight capacity during peak seasons."
        ]
    },
    {
        "filename": "08_Import_Export_Regulations.pdf",
        "title": "Import and Export Trade Compliance Controls",
        "category": "Import",
        "intro": "Global trade requires compliance with multiple international regulations. Non-compliance can lead to shipment seizures, criminal prosecutions, and heavy fines. This guide reviews trade controls, import licenses, and sanctions screening.",
        "sections": [
            ("Export Control & Dual-Use Goods", [
                "Export Administration Regulations (EAR) and International Traffic in Arms Regulations (ITAR) control the export of dual-use technologies and military goods.",
                "Export classification numbers (ECCN) must be declared on export filings to determine licensing requirements.",
            ]),
            ("Sanctions and Import Licensing", [
                "Sanctioned country and denied-party screenings check transactions against restricted entities lists.",
                "Import licenses are required for restricted goods, including agricultural commodities, pharmaceutical items, and hazardous chemicals.",
            ])
        ],
        "table_data": [
            ["ECCN Declaration", "EAR dual-use checks", "Zero licensing violations", "Mandatory on export filings"],
            ["Sanctions Screening", "Denied-party checks", "100% screening rate", "Prevents trading with embargoes"],
            ["Import Licenses", "Special agency approvals", "Valid pre-arrival permits", "Avoids customs confiscation"],
            ["Origin Rules", "FTA tariff criteria", "Verified certification logs", "Saves import duty payments"]
        ],
        "bullets": [
            "Screen all international customers, forwarders, and partners against global sanction lists prior to order confirmation.",
            "Identify the correct ECCN and export licensing requirements for all technical equipment and software products.",
            "Validate that certificates of origin conform to the specific rules of origin outlined in local trade agreements.",
            "Train logistics personnel annually on anti-bribery policies and compliance laws.",
            "Retain all trade compliance, audit trail, and screening records for at least 5 years."
        ]
    },
    {
        "filename": "09_Transportation_Modes_Comparison.pdf",
        "title": "Freight Transportation Modes Comparison",
        "category": "Transportation",
        "intro": "Selecting the correct transport mode is a balance between speed, cost, weight, distance, environmental impact, and cargo sensitivity. Multimodal strategies combine options to optimize global lanes.",
        "sections": [
            ("Operational Mode Characteristics", [
                "Air Freight: Fastest transit times (1-3 days), highest costs, best for high-value and perishable cargo.",
                "Ocean Freight: Lowest cost per ton-mile, suitable for large volumes, longest transit times (15-45 days).",
                "Rail Freight: Cost-effective for long inland distances, lower emissions than road transport.",
                "Road Freight: Highly flexible, handles door-to-door delivery, subject to fuel price volatility and road traffic.",
            ]),
            ("Intermodal and Multimodal Shipping", [
                "Intermodal transport utilizes standardized containers (ISO containers) across multiple modes without cargo handling.",
                "Multimodal transport combines modes (e.g., Sea-Air) under a single contract to optimize speed and cost.",
            ])
        ],
        "table_data": [
            ["Air Freight", "Fastest transit (1-3 days)", "Highest rate per kg", "High-value / Perishables"],
            ["Ocean Freight", "Slowest transit (15-45 days)", "Lowest rate per ton", "Bulky / Raw materials"],
            ["Rail Freight", "Medium speed (7-15 days)", "Inland bulk efficiency", "Inland container flows"],
            ["Road Freight", "Flexible routing (1-5 days)", "Door-to-door transit", "Last-mile distribution"]
        ],
        "bullets": [
            "Evaluate sea-air multimodal services to reduce transit times by 40% compared to pure ocean freight at a lower cost than air freight.",
            "Optimize container packaging using load planning software to maximize volumetric capacity.",
            "Utilize rail transport for inland freight over 500 miles to reduce carbon emissions by up to 75%.",
            "Establish contract rates with trucking firms to hedge against spot-market rate fluctuations.",
            "Verify terminal container handling weight limits when shipping heavy machinery or minerals."
        ]
    },
    {
        "filename": "10_Hazardous_Cargo_Compliance.pdf",
        "title": "Hazardous Materials and Dangerous Goods Handling",
        "category": "Compliance",
        "intro": "The transport of dangerous goods is heavily regulated to protect human life, property, and the environment. Compliance with international standards is mandatory across all transport modes.",
        "sections": [
            ("Regulatory Frameworks and UN Numbers", [
                "IMDG Code governs ocean transport, IATA Dangerous Goods Regulations (DGR) govern air transport, and ADR governs road transport.",
                "All hazardous materials must be assigned a 4-digit UN Number (e.g., UN 3480 for Lithium-ion batteries) and classified into one of nine hazard classes.",
            ]),
            ("Packaging and Placarding Rules", [
                "Dangerous goods must be packed in certified UN packaging and clearly marked with appropriate warning labels.",
                "Safety Data Sheets (SDS) must accompany all bookings, detailing chemical compositions and emergency response guidelines.",
            ])
        ],
        "table_data": [
            ["IMDG Code", "Maritime hazard classes", "Zero vessel accidents", "Regulates dangerous ocean cargo"],
            ["UN Number", "4-digit class code", "100% document accuracy", "Identifies specific chemicals"],
            ["UN Packaging", "Certified container boxes", "Passed pressure/drop tests", "Prevents chemical leaks"],
            ["SDS Sheet", "16-section document", "Accompanies all shipments", "Guides emergency responses"]
        ],
        "bullets": [
            "Ensure all employees involved in packing or loading dangerous goods hold current certification.",
            "Verify vessel stowage compatibility rules to prevent loading incompatible chemicals in adjacent bays.",
            "Attach clear, weather-resistant hazard placards to all four sides of shipping containers.",
            "Limit state-of-charge (SoC) for lithium-ion batteries to 30% during air transport compliance audits.",
            "Maintain emergency contact numbers on all hazardous shipping papers."
        ]
    },
    {
        "filename": "11_Inventory_Control_Management.pdf",
        "title": "Inventory Management and Optimization Standards",
        "category": "Inventory",
        "intro": "Inventory management balances the cost of holding inventory against the cost of stockouts and ordering. Optimizing this process ensures high customer service levels and healthy cash flow.",
        "sections": [
            ("Economic Order Quantity (EOQ)", [
                "EOQ is the optimal order quantity that minimizes total inventory costs, balancing holding costs (carrying cost) and ordering costs.",
                "Formula: EOQ = sqrt((2 * Demand * Ordering Cost) / Holding Cost). Safety stocks are added to account for demand and lead-time variability.",
            ]),
            ("Inventory Classification & Auditing", [
                "ABC Analysis categorizes items by value: A-items (high value, low quantity), B-items (medium value/quantity), and C-items (low value, high quantity).",
                "Cycle counting involves continuous audits of inventory parts throughout the year, avoiding warehouse shutdowns.",
            ])
        ],
        "table_data": [
            ["EOQ Model", "Balancing cost parameters", "Minimize total cost", "Maintains optimal inventory levels"],
            ["Safety Stock", "Lead-time buffer stock", "Stockout rate < 1%", "Covers delivery delay spikes"],
            ["ABC Analysis", "Value-based categorization", "Focused resource allocation", "Prioritizes A-class inventory"],
            ["Cycle Counting", "Continuous physical audits", "Inventory accuracy > 99%", "Replaces annual warehouse shut-downs"]
        ],
        "bullets": [
            "Recalculate safety stock levels quarterly to reflect carrier lead-time fluctuations and demand shifts.",
            "Apply tight inventory controls and daily tracking to all high-value Class A items.",
            "Use cycle counting to audit high-velocity stock locations weekly.",
            "Integrate warehouse management systems (WMS) with ERP platforms for real-time stock updates.",
            "Collaborate with suppliers using Vendor Managed Inventory (VMI) strategies to reduce storage overhead."
        ]
    },
    {
        "filename": "12_Delivery_Delay_Mitigation.pdf",
        "title": "Transit Delay Mitigation and Rerouting",
        "category": "Delivery Delay",
        "intro": "Logistics operations are prone to transit delays due to port congestion, bad weather, carrier blank sailings, and custom holds. Implementing contingency plans and rerouting protocols minimizes delay risks.",
        "sections": [
            ("Analysis of Delay Drivers", [
                "Primary delay drivers include port dwell time spikes, container terminal congestion, and custom clearance holds.",
                "Carrier blank sailings (canceled voyages) and weather-related maritime route adjustments delay container arrivals.",
            ]),
            ("Rerouting and Multimodal Alternatives", [
                "Dynamic rerouting diverts shipments from congested primary ports to less-crowded adjacent regional ports.",
                "Expedited land-bridge transport and ocean-rail multimodal corridors bypass sea passage bottlenecks.",
            ])
        ],
        "table_data": [
            ["Port Dwell Spikes", "Inland transport issues", "Evacuate within 48 hours", "Mitigates port congestion"],
            ["Blank Sailings", "Carrier capacity cuts", "Book space 4 weeks early", "Secures container space"],
            ["Dynamic Rerouting", "Route diversion protocols", "Diverts around bottlenecks", "Maintains delivery schedule"],
            ["SLA Buffers", "Transit time estimates", "Covers late-arrival risks", "Avoids retail stockout fines"]
        ],
        "bullets": [
            "Analyze historical carrier performance data before signing long-term space commitments.",
            "Set up automated email alerts in the TMS to flag container transshipments that exceed 5 days at hub ports.",
            "Maintain contracts with secondary regional ports to allow fast container rerouting.",
            "Establish sea-air transport combinations to speed up high-value delayed cargo.",
            "Add safety buffers into customer delivery SLAs to avoid late-delivery penalties."
        ]
    },
    {
        "filename": "13_Supply_Chain_Cost_Optimization.pdf",
        "title": "Landed Cost and Surcharge Optimization",
        "category": "Costs",
        "intro": "Landed cost represents the total price of a product once it has arrived at the buyer's door. Optimizing landed cost requires analyzing all logistics fees, custom tariffs, and handling charges.",
        "sections": [
            ("Landed Cost Components", [
                "Landed cost consists of the product purchase price, international freight rates, transport insurance, import customs duties, and local port drayage fees.",
                "Hidden surcharges include bunker adjustments, peak season surcharges, terminal handling charges (THC), and inland fuel multipliers.",
            ]),
            ("Cost Optimization Strategies", [
                "LCL-to-FCL cargo consolidation combines small shipments to lower per-unit transport costs.",
                "Automated freight invoice auditing checks carrier bills against contracted tariffs to catch duplicate items.",
            ])
        ],
        "table_data": [
            ["Landed Cost", "All-inclusive import cost", "Calculated per product unit", "Guides product pricing"],
            ["Consolidation", "LCL to FCL container packing", "Reduce rates by 25%", "Improves shipping efficiency"],
            ["Invoice Audits", "Tariff checking systems", "Identify billing mistakes", "Recovers carrier overcharges"],
            ["Free Time", "Extended port storage agreements", "14 days target", "Bypasses demurrage costs"]
        ],
        "bullets": [
            "Use container loading software to maximize weight and space utilization on all FCL shipments.",
            "Negotiate extended port free time from carriers to reduce risk of demurrage fees.",
            "Implement automated billing audits to identify billing mistakes and duplicate carrier invoices.",
            "Consolidate multiple small shipments from the same origin region into full container loads.",
            "Select import gateways with lower terminal handling fees and faster customs processes."
        ]
    },
    {
        "filename": "14_Customs_Valuation_HS_Tariffs.pdf",
        "title": "Customs Valuation and Tariff Standards",
        "category": "Customs",
        "intro": "Accurate customs valuation is essential to calculate correct import duties and taxes. Misdeclaring the value of goods can result in customs audits, seizure of cargo, and severe legal penalties.",
        "sections": [
            ("WTO Customs Valuation Methods", [
                "The transaction value is the primary valuation method, representing the price paid or payable for the goods when sold for export.",
                "Alternative valuation methods include transaction value of identical goods, transaction value of similar goods, deductive value, and computed value.",
            ]),
            ("Tariff Classifications & Customs Audits", [
                "The General Rules for Interpretation (GRI) govern the classification of complex, mixed, or disassembled items under correct HS tariff headings.",
                "Bilateral trade concessions under Free Trade Agreements require proof of origin compliance through certification.",
            ])
        ],
        "table_data": [
            ["Transaction Value", "Primary valuation rule", "100% matching invoice value", "Base for import duty calculation"],
            ["GRI Rules", "Tariff classification guide", "Uniform HS code selection", "Avoids duty disputes"],
            ["FTA Concessions", "Preferential duty rates", "Proof of origin required", "Lowers tariff payments"],
            ["Customs Audits", "Review of import records", "Audits up to 5 years back", "Ensures valuation compliance"]
        ],
        "bullets": [
            "Declare all additional payments made to suppliers, such as mold costs or design royalties, on customs declarations.",
            "Utilize the first-sale rule where legal to base import valuation on the price paid to the primary manufacturer.",
            "Validate origin certifications before declaring preferential duty rates under FTAs.",
            "Perform internal valuation audits quarterly to verify customs entries against general ledgers.",
            "Secure binding classification rulings from customs authorities for technical or custom-made items."
        ]
    },
    {
        "filename": "15_Global_Logistics_Corridors.pdf",
        "title": "Global Trade Corridors and Shipping Lanes",
        "category": "Supply Chain",
        "intro": "Global shipping lanes and trade corridors are the backbones of international trade. Understanding volume trends, passage bottleneck risks, and infrastructure capacities is essential for supply chain planning.",
        "sections": [
            ("Major Maritime Lanes & Bottlenecks", [
                "Transpacific shipping routes connect manufacturing hubs in East Asia to import gateways in North America.",
                "Asia-Europe shipping lanes utilize the Suez Canal to connect Asian factories directly to European markets.",
                "Passage bottlenecks include the Malacca Strait, the Suez Canal, and the Panama Canal, where disruptions trigger global delays.",
            ]),
            ("Inland Dedicated Freight Corridors", [
                "Inland trade corridors utilize dedicated rail corridors and express highways to connect landlocked regions to ocean terminals.",
                "Dry ports (inland container depots) act as inland customs clearance points to reduce seaside congestion.",
            ])
        ],
        "table_data": [
            ["Transpacific Lane", "East Asia to US West Coast", "Highest container volume", "Subject to port labor delays"],
            ["Asia-Europe Lane", "Suez Canal routing", "Handles 35% of containers", "Vulnerable to security blockages"],
            ["Malacca Strait", "Passage between Indian/Pacific", "Critical fuel transit lane", "Vulnerable to maritime piracy"],
            ["Dry Ports", "Inland container terminals", "Inland customs clearance", "Bypasses seaport yard congestion"]
        ],
        "bullets": [
            "Diversify maritime transport lanes to route cargo through both US East Coast and West Coast ports.",
            "Track water draft levels in the Panama Canal and security updates in the Suez Canal daily.",
            "Establish partnerships with dry ports to clear import cargo close to manufacturing sites.",
            "Monitor container availability at major Asian manufacturing ports during peak shipping seasons.",
            "Audit transshipment lane times to identify transit delay patterns at major hub terminals."
        ]
    },
    {
        "filename": "16_Trade_Corridor_JNPT_Rotterdam.pdf",
        "title": "Trade Corridor Briefing: JNPT to Rotterdam",
        "category": "Trade Corridors",
        "intro": "The maritime trade corridor connecting JNPT (Nhava Sheva) in India to the Port of Rotterdam in the Netherlands is a critical link between South Asia and Europe. Managing risks and scheduling along this route is essential for cargo punctuality.",
        "sections": [
            ("Transit Times & Maritime Routes", [
                "Standard ocean transit from JNPT to Rotterdam averages 24 to 28 days via the Suez Canal.",
                "Security risks in the Gulf of Aden and scheduling convoys through the Suez Canal can delay arrivals.",
            ]),
            ("European Import Compliance Rules", [
                "European Union customs regulations mandate Advanced Manifest filings (ENS) 24 hours prior to container loading at JNPT.",
                "Rotterdam terminal handling systems require pre-booked container transport appointments for inland rail or barge connection.",
            ])
        ],
        "table_data": [
            ["Ocean Transit", "Suez Canal maritime route", "24 to 28 transit days", "Subject to Suez convoy schedules"],
            ["EU Customs Filings", "ENS electronic manifest", "Submit 24h pre-loading", "Prevents custom cargo holds"],
            ["Rotterdam Yard", "Automated container terminals", "Pre-booked pickup slots", "Coordinates inland barge/rail"],
            ["Security Risks", "Gulf of Aden passage", "Carrier surcharge apply", "Vessel escort requirements check"]
        ],
        "bullets": [
            "Transmit ENS customs filings at least 36 hours prior to vessel loading at JNPT to allow for validation.",
            "Coordinate import drayage at Rotterdam in advance to secure container transport space.",
            "Track local monsoon patterns in India as they can impact container barge transport to JNPT.",
            "Use carrier services that offer alternative routing around Africa during Suez Canal security events.",
            "Audit terminal handling charges at both JNPT and Rotterdam annually to keep cost models current."
        ]
    },
    {
        "filename": "17_Trade_Corridor_Shanghai_Mundra.pdf",
        "title": "Trade Corridor Briefing: Shanghai to Mundra",
        "category": "Trade Corridors",
        "intro": "The trade lane from the Port of Shanghai in China to the Port of Mundra in India handles massive volumes of industrial parts, electronics, and chemical cargo. Managing customs clearances and weather risks is critical.",
        "sections": [
            ("Lanes Speed and Transit Challenges", [
                "Standard ocean transit from Shanghai to Mundra averages 14 to 18 days on direct loops.",
                "Heavy monsoons between June and September can limit vessel draft limits and delay berthing at Mundra.",
            ]),
            ("Customs Clearance & Documentation", [
                "Imports into India require electronic Bill of Entry filings via ICEGATE, including IGST payment verification.",
                "High customs inspection rates at Indian ports can delay container release times.",
            ])
        ],
        "table_data": [
            ["Direct Ocean loops", "Shanghai to Mundra", "14 to 18 transit days", "Direct shipping lines preferred"],
            ["Monsoon Disruptions", "June to September weather", "Draft limits may apply", "Adds 2-4 days weather delays"],
            ["ICEGATE Filing", "Bill of Entry registration", "Pre-arrival submission", "Speeds up customs release"],
            ["Inspection Audits", "Physical customs checks", "Required for specific HS classes", "Ensures compliance verification"]
        ],
        "bullets": [
            "Submit Bill of Entry documents via ICEGATE prior to vessel arrival at Mundra to speed up customs release.",
            "Track regional weather alerts and monsoon warnings in the Arabian Sea weekly during summer.",
            "Ensure all import shipping documents list the exact HS classifications to avoid customs inspection holds.",
            "Establish secondary cargo contracts with forwarders to secure cargo space during peak Chinese holidays.",
            "Audit customs agent performance quarterly to ensure on-time entry filing."
        ]
    },
    {
        "filename": "18_Port_Profile_Singapore_Hub.pdf",
        "title": "Port Infrastructure Profile: Port of Singapore",
        "category": "Port Profiles",
        "intro": "The Port of Singapore is the world's largest container transshipment hub, connecting shippers to over 600 ports in 120 countries. Its advanced automated infrastructure ensures rapid vessel turnaround times.",
        "sections": [
            ("Transshipment Operations & Tuas Port", [
                "As a major transshipment hub, Singapore specializes in transferring containers between international shipping lanes.",
                "The Tuas Port expansion project uses automated container yards, driverless cranes, and smart terminal systems to handle up to 65 million TEUs annually.",
            ]),
            ("Port Operations and Efficiency", [
                "Digital bunkering networks and smart scheduling keep average vessel turnaround times under 12 hours.",
                "Customs clearances utilize trade portals to authorize transshipments instantly.",
            ])
        ],
        "table_data": [
            ["Tuas Port Project", "Automated container hub", "65 million TEU capacity", "World's largest automated port"],
            ["Vessel Turnaround", "Smart vessel scheduling", "Average under 12 hours", "Maximizes carrier schedule efficiency"],
            ["Transshipment Speed", "Direct yard transfers", "Minimal dwell time", "Reduces shipping transit times"],
            ["Bunkering Network", "Digital marine fuel delivery", "100% flowmeter accuracy", "Reduces port refueling times"]
        ],
        "bullets": [
            "Select Singapore as the primary transshipment gateway for cargo routes between Asia and Europe.",
            "Utilize digital bunkering services in Singapore to ensure accurate marine fuel charging.",
            "Monitor transshipment container connection times to verify that forwarders meet connections.",
            "Incorporate Singapore port handling fee structures into global freight pricing calculators.",
            "Establish container buffer stocks at regional Singapore free trade zones to support rapid distribution."
        ]
    },
    {
        "filename": "19_HS_Chapter_84_85_Guide.pdf",
        "title": "HS Code Chapter 84 and 85 Classification Guide",
        "category": "HS Guides",
        "intro": "Chapters 84 and 85 of the Harmonized System (HS) tariff classification code cover machinery, mechanical appliances, electrical equipment, and telecommunication devices. Correct classification is critical as these chapters carry varied duty rates.",
        "sections": [
            ("Chapter 84: Mechanical Appliances", [
                "Chapter 84 covers nuclear reactors, steam boilers, industrial machinery, and mechanical parts.",
                "Classification depends on function, power source, and whether the machine performs a single function or multiple operations.",
            ]),
            ("Chapter 85: Electrical & Electronics", [
                "Chapter 85 covers electrical machinery, telecom equipment, audio-video devices, and electronic parts.",
                "Multi-component sub-assemblies require verification of essential character under General Rule of Interpretation 2(a).",
            ])
        ],
        "table_data": [
            ["Chapter 84", "Mechanical machinery/reactors", "Function-based classification", "Includes industrial pump systems"],
            ["Chapter 85", "Electrical/telecom devices", "Component character rules", "Includes microchip assemblies"],
            ["GRI 2(a) Rule", "Disassembled machine rule", "Classifies as complete item", "Guides import tariff valuation"],
            ["GRI 3(b) Rule", "Mixed component rule", "Essential character criteria", "Selects dominant HS code"]
        ],
        "bullets": [
            "Analyze detailed engineering drawings and function descriptions before assigning HS codes under Chapter 84.",
            "Apply GRI 2(a) to classify knocked-down or disassembled machinery under the finished product tariff heading.",
            "Ensure commercial invoices list separate values for electrical assemblies and mechanical housings to simplify audits.",
            "Validate product compliance certificates (e.g., CE, FCC) before declaring electronic goods at customs.",
            "Retain product classification rationale records for at least 5 years to support customs audit responses."
        ]
    },
    # ------------------ NEW DOCUMENTS (20-38) ------------------
    {
        "filename": "20_Air_Cargo_Freight_Operations.pdf",
        "title": "Air Cargo Freight Operations and IATA Standards",
        "category": "Freight",
        "intro": "Air cargo operations provide high-speed transportation services critical for time-sensitive, high-value, and perishable goods. Operating within this sector requires strict adherence to international safety, security, and operational standards.",
        "sections": [
            ("IATA Regulatory Framework", [
                "The International Air Transport Association (IATA) establishes global standards for airline safety, security, and cargo operations. This includes the Dangerous Goods Regulations (DGR) and Live Animals Regulations (LAR).",
                "Air Waybills (AWB) serve as the non-negotiable contract of carriage between the shipper and the carrier, detailing cargo weight, volume, routing, and description.",
            ]),
            ("Air Cargo Terminal Handling", [
                "Cargo terminals utilize Unit Load Devices (ULDs) such as aircraft pallets and containers to consolidate cargo. Handling processes include receiving, security screening, ULD buildup, and ramp transfer.",
                "Pre-load security screening uses advanced X-ray systems, explosive trace detection (ETD), and physical inspections to prevent security threats.",
            ])
        ],
        "table_data": [
            ["IATA DGR", "Dangerous goods guidelines", "100% compliance rate", "Prevents safety incidents on board"],
            ["Air Waybill (AWB)", "Contract of carriage document", "Electronic E-AWB standard", "Enables digital freight tracking"],
            ["Unit Load Device (ULD)", "Standard aircraft container", "Proper weight distribution", "Maximizes aircraft hold space"],
            ["Cargo Screening", "X-ray and ETD systems", "Zero un-screened cargo", "Ensures compliance with aviation laws"]
        ],
        "bullets": [
            "Use electronic Air Waybills (e-AWB) to reduce processing errors and speed up cargo acceptance at terminals.",
            "Verify ULD weight limits and balance rules before completing load planning.",
            "Perform regular audits of cargo terminal agents to ensure compliance with temperature storage rules.",
            "Consolidate small shipments to utilize standard volume-based pricing discounts.",
            "Establish secondary airport routing plans to bypass major air hub congestion."
        ]
    },
    {
        "filename": "21_Ocean_Freight_Operations.pdf",
        "title": "Ocean Freight Operations and Maritime Shipping",
        "category": "Freight",
        "intro": "Ocean freight is the primary mode of international trade, handling over 80% of global trade volumes. Standardized processes, containerization, and carrier alliances drive maritime shipping efficiency.",
        "sections": [
            ("Containerized Shipping (FCL vs LCL)", [
                "Full Container Load (FCL) shipping reserves an entire container for a single shipper. This reduces damage risks and speeds up transit times.",
                "Less than Container Load (LCL) shipping consolidates cargo from multiple shippers into a single container. LCL requires container consolidation at origin and deconsolidation at destination, increasing transit times and risk.",
            ]),
            ("Ocean Carrier Alliances and Lanes", [
                "Major global container lines operate within strategic alliances (e.g., 2M, Ocean Alliance, THE Alliance) to share vessel capacity and optimize routes.",
                "Ocean shipping lanes are divided into major East-West trade loops (Transpacific, Transatlantic, Asia-Europe) and regional feeder services.",
            ])
        ],
        "table_data": [
            ["FCL Shipping", "Exclusive container use", "Faster direct transit", "Ideal for high-volume shipments"],
            ["LCL Consolidation", "Shared container space", "Cost-effective for small volumes", "Requires CFS handling time"],
            ["Carrier Alliances", "Vessel sharing agreements", "Broad lane coverage", "Increases sailing frequency options"],
            ["Feeder Services", "Small container vessels", "Connects regional ports", "Links minor terminals to hubs"]
        ],
        "bullets": [
            "Book FCL space at least 3 to 4 weeks prior to cargo ready dates during peak shipping seasons.",
            "Inspect container seals and structural conditions before loading cargo at shipper facilities.",
            "Audit Ocean Bill of Lading descriptions to verify cargo weights match customs filings.",
            "Negotiate container detention and demurrage terms directly with carriers during annual contract reviews.",
            "Utilize container track-and-trace platforms to receive real-time updates on vessel locations."
        ]
    },
    {
        "filename": "22_Rail_Freight_Transportation.pdf",
        "title": "Rail Freight Transportation and Dedicated Corridors",
        "category": "Freight",
        "intro": "Rail freight provides high-capacity, energy-efficient transportation for bulk cargo, finished goods, and intermodal containers over long inland distances. Dedicated rail corridors connect major industrial regions directly to sea terminals.",
        "sections": [
            ("Dedicated Freight Corridors (DFCs)", [
                "Dedicated Freight Corridors are specialized rail systems designed solely for freight trains. This separates them from passenger train traffic, increasing speed, reliability, and load capacities.",
                "Double-stack container trains operate on DFCs, doubling carrying capacity per train and lowering per-container transport costs.",
            ]),
            ("Rail Terminal Operations", [
                "Intermodal rail terminals utilize reach stackers, gantry cranes, and automated tracking to transfer containers between rail cars and trucks.",
                "Rail schedules operate on fixed loops, requiring coordinated drayage dispatches to avoid storage penalties.",
            ])
        ],
        "table_data": [
            ["Dedicated Corridors", "Freight-only rail tracks", "Average speeds >60 km/h", "Bypasses passenger train delays"],
            ["Double-Stack Trains", "Two-tier container loading", "Double train capacity", "Reduces per-unit transport costs"],
            ["Intermodal Terminal", "Container transfer yards", "Rapid gantry turnaround", "Coordinates rail-to-truck moves"],
            ["Rail Freight Rate", "Fixed rate-per-kilometer", "Stable price structure", "Hedges against diesel price hikes"]
        ],
        "bullets": [
            "Use dedicated rail corridors for inland cargo moves exceeding 400 miles to reduce fuel costs.",
            "Verify rail container locking systems to ensure cargo remains secure during transit vibrations.",
            "Coordinate drayage carriers to pick up containers within the terminal's free-time window.",
            "Incorporate rail freight schedules into warehouse shipment planning systems.",
            "Audit terminal handling bills to ensure accurate drayage and storage pricing."
        ]
    },
    {
        "filename": "23_Road_Freight_Transportation.pdf",
        "title": "Road Freight Transportation and Fleet Operations",
        "category": "Freight",
        "intro": "Road transportation is a critical logistics component, providing door-to-door flexibility. Fleet operations must manage driver schedules, fuel costs, vehicle maintenance, and route planning.",
        "sections": [
            ("Full Truckload (FTL) and LTL Operations", [
                "Full Truckload (FTL) transport dedicates an entire trailer to a single customer, providing direct transit and minimal handling.",
                "Less-than-Truckload (LTL) consolidates shipments from multiple customers into a single trailer, utilizing regional hub-and-spoke networks to lower costs for small loads.",
            ]),
            ("Fleet Routing & Drivers Scheduling", [
                "Fleet routing software uses GPS, traffic models, and cargo parameters to plan optimal driving routes.",
                "Driver scheduling must comply with Hours of Service (HoS) regulations, requiring electronic logging devices (ELDs) to monitor drive times.",
            ])
        ],
        "table_data": [
            ["FTL Operations", "Dedicated trailer capacity", "Direct origin-to-destination", "Minimal cargo damage risk"],
            ["LTL Networks", "Hub-and-spoke sorting", "Consolidated delivery", "Lower costs for small shipments"],
            ["HoS Regulations", "Driver drive-time limits", "100% ELD compliance", "Ensures highway safety compliance"],
            ["Fleet Telematics", "GPS and engine tracking", "Real-time route optimization", "Reduces vehicle idling times"]
        ],
        "bullets": [
            "Deploy LTL consolidation programs to combine smaller branch deliveries into FTL runs.",
            "Equip all trucks with ELDs to ensure accurate driver time tracking and regulatory compliance.",
            "Establish preventative fleet maintenance programs to minimize on-road vehicle breakdowns.",
            "Implement telematics software to optimize driver routing and reduce fuel waste.",
            "Audit trucking carrier compliance papers annually to confirm active liability insurance."
        ]
    },
    {
        "filename": "24_Last_Mile_Delivery_Logistics.pdf",
        "title": "Last Mile Delivery Logistics and Urban Routing",
        "category": "Logistics",
        "intro": "Last mile delivery is the final leg of the supply chain, moving goods from local hubs to final buyers. It is the most complex, costly, and consumer-visible portion of the logistics chain.",
        "sections": [
            ("Urban Logistics Challenges", [
                "Urban areas feature traffic congestion, parking restrictions, narrow roads, and strict emission zones.",
                "E-commerce has driven demand for fast, on-time deliveries, leading to higher order volumes and smaller package sizes.",
            ]),
            ("Urban Routing Technologies", [
                "Dynamic routing software calculates delivery routes based on real-time traffic conditions, delivery windows, and package types.",
                "Urban distribution micro-hubs allow carriers to place inventory closer to high-density delivery zones.",
            ])
        ],
        "table_data": [
            ["Micro-Hubs", "Local urban sorting depots", "Reduce delivery loop distances", "Enables cargo-bike deliveries"],
            ["Dynamic Routing", "Real-time navigation software", "Optimizes driver stop sequences", "Increases deliveries per hour"],
            ["Emission Zones", "Restricted access zones", "Electric van fleets required", "Complies with urban laws"],
            ["Delivery Tracking", "SMS and GPS updates", "Real-time package locations", "Reduces failed delivery attempts"]
        ],
        "bullets": [
            "Establish urban micro-hubs to decrease last-mile delivery distances and lower emissions.",
            "Utilize local pickup lockers to consolidate package drops and reduce failed deliveries.",
            "Deploy electric delivery vehicles in city centers to bypass emission fees and zones.",
            "Equip customer portals with real-time driver tracking links to improve delivery success rates.",
            "Benchmark delivery carrier performance weekly against on-time delivery metrics."
        ]
    },
    {
        "filename": "25_Reverse_Logistics_Operations.pdf",
        "title": "Reverse Logistics and Returns Management",
        "category": "Logistics",
        "intro": "Reverse logistics covers the processing of products returning from consumers back through the supply chain. Efficient returns handling recovers value, reduces waste, and maintains customer loyalty.",
        "sections": [
            ("The Return Flow Process", [
                "The returns process includes customer return initiation, transport back to a facility, inspection, and disposition decision-making.",
                "Product dispositions include return-to-stock, repackaging, refurbishment, liquidation, recycling, or disposal.",
            ]),
            ("Value Recovery Strategies", [
                "Refurbishment updates and tests returned items to allow resale on secondary markets.",
                "Automated returns processing systems quickly identify product conditions to reduce depot hold times.",
            ])
        ],
        "table_data": [
            ["Returns Depot", "Dedicated sorting facility", "Inspect returns within 24h", "Speeds up customer refunds"],
            ["Product Disposition", "Sorting category rules", "Consistent sorting decisions", "Maximizes recovery values"],
            ["Refurbishment", "Product repair and testing", "Restores products to like-new", "Allows secondary resale"],
            ["Recycling / Disposal", "E-waste and scrap rules", "Zero landfill waste target", "Complies with environmental laws"]
        ],
        "bullets": [
            "Establish clear returns policies and customer portals to make returns drop-offs simple.",
            "Optimize returns inspection workflows to sort sellable items back into stock quickly.",
            "Monitor product returns trends to identify recurring manufacturing defects.",
            "Collaborate with liquidators to clear out returns that cannot be resold directly.",
            "Audit return transport costs to identify opportunities for consolidating return shipping."
        ]
    },
    {
        "filename": "26_Cargo_Marine_Insurance_Risk.pdf",
        "title": "Cargo and Marine Insurance Risk Management",
        "category": "Risk",
        "intro": "Cargo and marine insurance protects businesses from financial losses due to cargo damage, theft, vessel accidents, or transit delays. Proper risk planning and coverage options are essential.",
        "sections": [
            ("Types of Cargo Coverages", [
                "All-Risk Policies: Provide broad coverage for physical loss or damage from external causes, subject to exclusions.",
                "Named Perils Policies: Cover only the specific risks listed in the policy, such as collision, sinking, or fire.",
                "General Average: A maritime law principle where all cargo owners share the costs of saving a vessel during a crisis.",
            ]),
            ("Claims & Risk Management", [
                "Filing a claim requires proof of damage, a bill of lading, a commercial invoice, and a packing list.",
                "Risk planning includes auditing carrier liability limits and using proper cargo loading methods.",
            ])
        ],
        "table_data": [
            ["All-Risk Policy", "Broad cargo protection", "Minimal exclusion lists", "Best for high-value cargo"],
            ["General Average", "Vessel crisis cost sharing", "Mandatory cargo contributions", "Avoids container seizures"],
            ["Claims Filing", "Damage documentation", "Submit within 3 days", "Ensures fast claims reviews"],
            ["Carrier Liability", "Statutory carrier limits", "Often insufficient for full value", "Requires supplemental insurance"]
        ],
        "bullets": [
            "Ensure cargo insurance coverage is active for 110% of the combined cargo and freight value.",
            "Document all cargo arrivals with high-resolution photos if package damage is visible.",
            "Instruct logistics teams on how to handle General Average declarations by carriers.",
            "Audit shipper cargo packing methods to ensure they meet transit protection guidelines.",
            "Review cargo insurance policy exclusions annually to address new route risks."
        ]
    },
    {
        "filename": "27_Dangerous_Goods_Compliance.pdf",
        "title": "Dangerous Goods Compliance and Safety Standards",
        "category": "Compliance",
        "intro": "Transporting dangerous goods requires compliance with strict safety regulations. Implementing correct packaging, labeling, and handling procedures prevents accidents and ensures regulatory compliance.",
        "sections": [
            ("Hazard Classifications", [
                "Dangerous goods are divided into nine classes, including explosives, gases, flammable liquids, toxic materials, and radioactive substances.",
                "Each class has specific requirements for packaging, stowage, segregation, and emergency response.",
            ]),
            ("Safety Controls & Training", [
                "Workers handling dangerous goods must complete regular safety training programs.",
                "Emergency response plans must detail chemical spills control, worker safety gear, and emergency contact lists.",
            ])
        ],
        "table_data": [
            ["Hazard Class 3", "Flammable liquids", "Strict temperature limits", "Includes fuel and solvents"],
            ["Hazard Class 9", "Miscellaneous dangers", "Lithium battery rules apply", "Includes magnet materials"],
            ["Compliance Audit", "Safety documents review", "Zero compliance errors", "Prevents regulatory fines"],
            ["Emergency Response", "Spill kits and safety gear", "Instant emergency calls", "Protects terminal facilities"]
        ],
        "bullets": [
            "Conduct safety audits of dangerous goods storage yards every six months.",
            "Verify all chemical shipments are accompanied by up-to-date Safety Data Sheets.",
            "Ensure transport vehicles carry appropriate warning placards before dispatching cargo.",
            "Keep emergency contact numbers clearly printed on all shipping documents.",
            "Establish training schedules to keep worker safety certifications active."
        ]
    },
    {
        "filename": "28_Container_Types_Logistics.pdf",
        "title": "ISO Container Types and Shipping Standards",
        "category": "Logistics",
        "intro": "Standardized ISO containers are the foundation of global multimodal transport. Utilizing the correct container type protects cargo and optimizes shipping rates.",
        "sections": [
            ("Standard Dry and Specialized Containers", [
                "Dry Vans (20ft and 40ft): General-purpose containers used for dry commodities and boxed goods.",
                "Reefers: Temperature-controlled containers equipped with active cooling units for food and pharmaceuticals.",
                "Open Tops and Flat Racks: Specialized containers used for over-height, over-width, or heavy machinery cargo.",
            ]),
            ("Container Specifications & Limits", [
                "Container tare weight, gross weight limits, and internal dimensions must be verified before loading.",
                "Cargo weight must be distributed evenly across the container floor to prevent damage during handling.",
            ])
        ],
        "table_data": [
            ["20ft Dry Van", "Standard general container", "Max payload ~21,800 kg", "Ideal for heavy, dense cargo"],
            ["40ft Dry Van", "Standard high-volume container", "Max payload ~26,700 kg", "Best for light, bulky cargo"],
            ["Reefer Container", "Temperature controlled", "Range -30°C to +30°C", "Requires continuous electricity"],
            ["Flat Rack Container", "Rigid folding end-walls", "Open-sided loading", "Used for oversized machinery"]
        ],
        "bullets": [
            "Verify container payload weight limits before starting heavy cargo load planning.",
            "Use desiccants inside dry containers to prevent moisture damage during sea voyages.",
            "Secure heavy machinery cargo to flat rack anchor points using steel chains.",
            "Audit reefer temperature setpoints before handing over containers to carriers.",
            "Confirm container structural inspections (CSC safety plates) are active before loading."
        ]
    },
    {
        "filename": "29_Port_Security_Profiles.pdf",
        "title": "Port Security Standards and ISPS Code Compliance",
        "category": "Port Profiles",
        "intro": "Port security is regulated under international treaties to protect maritime infrastructure from security threats. Implementing access controls, cargo screening, and emergency plans is mandatory.",
        "sections": [
            ("The ISPS Security Code", [
                "The International Ship and Port Facility Security (ISPS) Code is an IMO amendment that establishes security standards for ships and ports.",
                "ISPS requirements include Port Facility Security Assessments (PFSA) and Port Facility Security Plans (PFSP).",
            ]),
            ("Port Access Control and Screening", [
                "Terminal gates utilize biometric access controls, license plate readers, and security checks to monitor vehicles.",
                "Cargo scanning systems scan shipping containers for unauthorized goods and radioactive materials.",
            ])
        ],
        "table_data": [
            ["ISPS Code Level 1", "Normal operational state", "Standard security checks", "Default security level"],
            ["ISPS Code Level 2", "Increased security threat", "Additional security guards", "Triggered by general alerts"],
            ["ISPS Code Level 3", "Imminent security risk", "Terminal access closed", "Controlled by national forces"],
            ["Container Scanning", "Radiation and X-ray systems", "Scans high-risk containers", "Prevents illegal imports"]
        ],
        "bullets": [
            "Audit port facility compliance reports to verify active ISPS certifications.",
            "Implement biometric access gates at terminal entries to secure port perimeters.",
            "Establish cargo screening partnerships with customs agencies to reduce scanning delays.",
            "Conduct quarterly port emergency exercises to test security responses.",
            "Review terminal security plans annually to address new cyber and physical risks."
        ]
    },
    {
        "filename": "30_Fleet_Management_Carrier.pdf",
        "title": "Fleet Management and Carrier Operations",
        "category": "Carrier Selection",
        "intro": "Fleet management involves coordinating commercial vehicles to optimize fuel efficiency, maintain safety compliance, and ensure reliable carrier operations.",
        "sections": [
            ("Vehicle Telematics and Maintenance", [
                "GPS tracking, engine diagnostics, and driver behavior telemetry are monitored in real-time.",
                "Preventative maintenance schedules reduce roadside breakdowns and extend vehicle operating lives.",
            ]),
            ("Route Optimization and Safety", [
                "Dynamic routing systems adjust truck runs based on traffic delays and delivery windows.",
                "Safety policies must regulate driver rest breaks, speed limits, and cargo securement methods.",
            ])
        ],
        "table_data": [
            ["Fleet Telematics", "Real-time vehicle tracking", "Improves routing efficiency", "Reduces fleet fuel consumption"],
            ["Preventative Care", "Scheduled truck servicing", "Reduces roadside breakdowns", "Extends fleet service life"],
            ["Dynamic Routing", "Traffic-aware route planning", "Meets delivery time targets", "Optimizes driver stop lists"],
            ["Safety Standards", "Speed and rest regulations", "Decreases accident rates", "Ensures compliance with laws"]
        ],
        "bullets": [
            "Verify carrier fleet safety records before signing transport service agreements.",
            "Implement automatic diagnostic alerts on all vehicles to catch engine issues early.",
            "Train truck drivers in defensive driving methods to lower fuel consumption.",
            "Utilize route optimization software to reduce empty truck return miles.",
            "Ensure cargo securement straps are inspected regularly to prevent transit damage."
        ]
    },
    {
        "filename": "31_Sustainability_Risk.pdf",
        "title": "Supply Chain Sustainability and Carbon Emissions",
        "category": "Risk",
        "intro": "Sustainable supply chain management focuses on reducing environmental impact, limiting carbon emissions, and building green logistics networks to meet corporate goals.",
        "sections": [
            ("Carbon Accounting and Regulations", [
                "Logistics carbon accounting calculates greenhouse gas emissions across scope 1, 2, and 3 activities.",
                "Environmental laws mandate emissions reduction targets and carbon pricing mechanisms in major trade regions.",
            ]),
            ("Green Logistics Strategies", [
                "Logistics operations adopt electric delivery fleets and biofuel-powered container ships to lower emissions.",
                "Consolidating shipments and using rail transport instead of road freight optimizes energy usage.",
            ])
        ],
        "table_data": [
            ["Scope 3 Emissions", "Supply chain carbon costs", "Carbon auditing standard", "Requires supplier carbon data"],
            ["Green Fleets", "Electric and hybrid trucks", "Zero urban emissions", "Bypasses city emission fees"],
            ["Rail Modal Shift", "Road to rail transport shift", "Reduces emissions by 75%", "Improves long-haul energy use"],
            ["Biofuel Shipping", "Biofuel container shipping", "Reduces marine carbon impact", "Supports green ocean lanes"]
        ],
        "bullets": [
            "Integrate carbon calculation models into transport management software.",
            "Collaborate with carriers that utilize fuel-efficient or alternative-power trucks.",
            "Assess supply chain routes to select paths with lower carbon footprints.",
            "Set emissions reduction targets for all warehouse and transport operations.",
            "Report scope 3 carbon emissions annually to maintain environmental compliance."
        ]
    },
    {
        "filename": "32_Warehouse_Automation_Warehousing.pdf",
        "title": "Warehouse Automation and Smart Systems",
        "category": "Warehousing",
        "intro": "Warehouse automation implements robotic picking systems, automated storage, and intelligent software to optimize throughput, inventory accuracy, and space utilization.",
        "sections": [
            ("Automated Storage and Retrieval Systems", [
                "ASRS uses computer-controlled cranes and shuttles to automatically place and retrieve loads from high-density racks.",
                "Autonomous Mobile Robots (AMRs) transport inventory goods across warehouse floors, reducing worker walking times.",
            ]),
            ("Warehouse Management Software", [
                "Warehouse Management Systems (WMS) optimize inventory storage, coordinate robot tasks, and track stock locations.",
                "Integrating WMS with ERP systems ensures on-time order processing and inventory accuracy.",
            ])
        ],
        "table_data": [
            ["ASRS Systems", "Automated crane racking", "Maximizes storage density", "Saves floor space"],
            ["Autonomous Mobile Robots", "AMR floor transport systems", "Reduces picking times", "Improves worker safety"],
            ["WMS Software", "Inventory tracking platform", "Inventory accuracy >99.9%", "Coordinates daily operations"],
            ["RFID Integration", "RFID sensor scanners", "Real-time stock updates", "Speeds up cargo receiving"]
        ],
        "bullets": [
            "Conduct warehouse site audits to identify optimal layout designs for AMR integration.",
            "Integrate automated barcode scanning gates at shipping docks to speed up verification.",
            "Optimize WMS picking paths to reduce order processing cycle times.",
            "Install preventative maintenance plans for automated conveyor and crane systems.",
            "Train warehouse workers on robotic system safety controls and operating guides."
        ]
    },
    {
        "filename": "33_Inventory_Optimization.pdf",
        "title": "Inventory Optimization and Demand Planning",
        "category": "Inventory",
        "intro": "Inventory optimization utilizes demand planning, lead-time predictions, and safety stock models to maintain high service levels while reducing inventory holding costs.",
        "sections": [
            ("Demand Forecasting Models", [
                "Demand planning analyzes sales history, seasonal trends, and market events to forecast inventory requirements.",
                "Machine learning algorithms improve forecast accuracy by processing external data, such as weather and economic indicators.",
            ]),
            ("Safety Stock and Lead-Time Management", [
                "Safety stock calculations must account for supplier lead-time delays and transport variations.",
                "Collaborative planning, forecasting, and replenishment (CPFR) structures align inventory plans between suppliers and retailers.",
            ])
        ],
        "table_data": [
            ["Demand Forecasting", "Statistical trend analysis", "Reduces forecast errors", "Informs production plans"],
            ["Safety Stock Buffer", "Lead-time variation cover", "Stockout rate <1%", "Covers delivery delays"],
            ["CPFR Strategy", "Supplier-retailer planning", "Optimizes supply capacity", "Reduces excess stocks"],
            ["Multi-Echelon Stocking", "Network stock optimization", "Balances regional inventories", "Lowers inventory holding costs"]
        ],
        "bullets": [
            "Adjust regional warehouse stocking levels based on seasonal sales forecasts.",
            "Recalculate safety stock margins monthly to reflect actual carrier delivery times.",
            "Use collaborative planning tools to share demand forecasts with key suppliers.",
            "Audit inventory turnover rates quarterly to identify slow-moving items.",
            "Deploy multi-echelon inventory optimization to balance stocks across regional hubs."
        ]
    },
    {
        "filename": "34_RFID_IoT_Logistics.pdf",
        "title": "RFID, IoT and Real-Time Tracking in Logistics",
        "category": "Logistics",
        "intro": "RFID and IoT technologies provide real-time visibility across supply chains. Utilizing smart sensors and tag readers allows businesses to track cargo location, temperature, and security status.",
        "sections": [
            ("RFID Tracking in Warehouses", [
                "Radio Frequency Identification (RFID) utilizes electromagnetic fields to automatically identify and track tags attached to objects.",
                "RFID gates scan entire pallets instantly, replacing manual barcode scans and speeding up cargo receiving.",
            ]),
            ("IoT Sensors and Telemetry", [
                "Internet of Things (IoT) sensors transmit real-time data, including GPS location, temperature, shock impact, and light exposure.",
                "Continuous telemetry data helps logistics managers detect cargo theft, damage events, and transit delays early.",
            ])
        ],
        "table_data": [
            ["RFID Pallet Scan", "Instant tag scanning gates", "Scans pallets in seconds", "Saves manual labor hours"],
            ["IoT GPS Tracker", "Continuous location tracking", "Real-time arrival ETAs", "Improves shipment visibility"],
            ["Shock Sensor Tags", "Impact monitoring sensors", "Identifies handling damage", "Supports damage claims"],
            ["BLE Telemetry", "Bluetooth data loggers", "Temperature tracking reports", "Ensures pharmaceutical safety"]
        ],
        "bullets": [
            "Equip high-value cargo containers with active IoT GPS trackers to monitor transit paths.",
            "Install RFID gates at warehouse entry and exit docks to automate inventory registration.",
            "Review IoT shock logs to pinpoint cargo handling damage points on routes.",
            "Verify BLE sensor battery lives before dispatching long-distance shipments.",
            "Integrate IoT tracking feeds directly into customer support dashboards."
        ]
    },
    {
        "filename": "35_AI_Logistics_Technology.pdf",
        "title": "Artificial Intelligence and Technology in Logistics",
        "category": "Logistics",
        "intro": "Artificial intelligence is transforming logistics by optimizing routes, predicting delivery delays, automating warehouse systems, and improving customs document audits.",
        "sections": [
            ("Predictive Machine Learning Models", [
                "Machine learning models predict transit delays by analyzing historic carrier performance, port dwell times, and weather data.",
                "Freight price pricing algorithms analyze demand patterns and fuel indices to calculate spot-market rates.",
            ]),
            ("Generative AI and Document Audits", [
                "Generative AI assistants analyze customs regulations, draft commercial invoices, and verify document compliance.",
                "Natural language processing systems automate customer support communications and trace shipment statuses.",
            ])
        ],
        "table_data": [
            ["Delay Predictions", "Predictive machine learning", "Identifies shipping delays", "Allows proactive rerouting"],
            ["AI Pricing Engine", "Dynamic rate calculations", "Optimizes freight spending", "Reduces spot-market costs"],
            ["Document Auditing", "AI text parsing engines", "Finds documentation errors", "Prevents custom cargo holds"],
            ["Customer Chatbots", "Generative AI assistants", "Automated tracking updates", "Reduces support workloads"]
        ],
        "bullets": [
            "Deploy predictive models to identify carriers with higher risks of delivery delays.",
            "Integrate dynamic pricing calculators into online customer booking systems.",
            "Use AI document parsers to audit commercial invoices before customs submission.",
            "Deploy AI chatbots to handle routine shipment tracking queries.",
            "Audit AI model accuracy metrics quarterly to prevent prediction drifts."
        ]
    },
    {
        "filename": "36_Customs_Brokerage_Risk.pdf",
        "title": "Customs Brokerage and Risk Management",
        "category": "Customs",
        "intro": "Customs brokers act as agents for importers to clear goods through customs. Managing customs compliance risks prevents cargo holds, penalty charges, and import license cancellations.",
        "sections": [
            ("Customs Broker Responsibilities", [
                "Customs brokers classify goods under HS codes, calculate duties and taxes, and submit electronic entries to customs agencies.",
                "Importers remain legally responsible for document accuracy, requiring active oversight of broker filings.",
            ]),
            ("Managing Customs Risks", [
                "Valuation disputes, origin verification failures, and misclassified HS codes are primary customs compliance risks.",
                "Post-clearance audits check past customs entries to verify compliance and correct duty payments.",
            ])
        ],
        "table_data": [
            ["Broker Filing", "Electronic customs entries", "100% filing accuracy", "Speeds up port clearance"],
            ["Customs Bonds", "Financial security guarantees", "Active bond coverage", "Ensures duty payments"],
            ["Valuation Audit", "Invoice verification checks", "Zero valuation discrepancies", "Prevents customs penalties"],
            ["Compliance Logs", "Filing audit record tracking", "5-year document retention", "Ensures audit readiness"]
        ],
        "bullets": [
            "Establish Standard Operating Procedures (SOPs) for customs brokers to ensure uniform HS code selection.",
            "Verify customs bond validity limits annually to ensure compliance coverage.",
            "Perform random audits of broker-filed customs declarations to verify accuracy.",
            "Ensure transaction value declarations conform to WTO valuation standards.",
            "Retain all broker communication and customs receipt records for at least 5 years."
        ]
    },
    {
        "filename": "37_Bills_of_Lading_Freight.pdf",
        "title": "Bills of Lading and Shipping Documentation",
        "category": "Freight",
        "intro": "The Bill of Lading (B/L) is a critical document in international trade. It acts as a contract of carriage, a receipt of cargo, and a document of title to the goods.",
        "sections": [
            ("Functions of the Bill of Lading", [
                "Contract of Carriage: Detailing terms, routing, and responsibilities of the carrier and shipper.",
                "Receipt of Cargo: Issued by the carrier once goods are loaded on the vessel, detailing quantity and condition.",
                "Document of Title: Transferable or non-negotiable formats (such as Seaway Bills or Telex Release) govern cargo release.",
            ]),
            ("Other Critical Shipping Documents", [
                "Air Waybills (AWBs) govern air freight carriage but do not act as documents of title.",
                "Packing Lists, Commercial Invoices, and Certificates of Origin must align with B/L details to clear customs.",
            ])
        ],
        "table_data": [
            ["Original B/L", "Negotiable document of title", "Must present to release cargo", "Required for bank LC terms"],
            ["Sea Waybill", "Non-negotiable receipt", "Instant cargo release", "Simplifies trusted transfers"],
            ["Telex Release", "Digital message release", "Speeds up destination release", "Bypasses courier delays"],
            ["Air Waybill (AWB)", "Non-negotiable air document", "Standard air freight contract", "Does not transfer title"]
        ],
        "bullets": [
            "Use Sea Waybills or Telex releases to bypass physical document courier delays at destination ports.",
            "Verify that cargo descriptions on Bills of Lading match commercial invoice text exactly.",
            "Store original Bills of Lading in secure locations as lost documents delay cargo release.",
            "Audit freight forwarder draft B/Ls before issuing final carrier instructions.",
            "Ensure port of discharge details match routing plans to avoid delivery delays."
        ]
    },
    {
        "filename": "38_Packaging_Standards_Compliance.pdf",
        "title": "Packaging Standards and Compliance",
        "category": "Compliance",
        "intro": "Proper packaging protects cargo from damage during handling, stacking, and shipping. Compliance with international packaging standards is mandatory for international trade.",
        "sections": [
            ("International Packaging Standards", [
                "ISPM 15 regulates wood packaging materials (WPM) to prevent the spread of forest pests, requiring heat treatment or fumigation.",
                "Palletization standards specify pallet dimensions, load stacking limits, and stretch wrapping requirements.",
            ]),
            ("Hazardous and Sensitive Cargo Packaging", [
                "Hazardous cargo requires certified UN packaging that has passed drop and pressure testing.",
                "Fragile and high-value cargo utilize shock-absorbing packaging materials and tilt indicators.",
            ])
        ],
        "table_data": [
            ["ISPM 15 Standard", "Wood fumigation standard", "Official stamp certification", "Prevents custom port holds"],
            ["Palletization", "Standardized pallet sizes", "Under maximum height limits", "Allows easy forklift handling"],
            ["UN Packaging", "Certified dangerous goods boxes", "Passed pressure/leak tests", "Prevents dangerous spills"],
            ["Shock Packaging", "Foam and tilt indicator tags", "Records rough handling", "Supports damage claims"]
        ],
        "bullets": [
            "Ensure all wood pallets carry the official ISPM 15 compliance stamp before exporting cargo.",
            "Optimize pallet stacking arrangements to prevent package crushing in containers.",
            "Use certified UN-packaging boxes for all Class 9 and other dangerous goods shipments.",
            "Perform drop and vibration tests on new product packaging designs before starting volume shipping.",
            "Audit warehouse packaging operations monthly to verify compliance with standards."
        ]
    }
]

def main():
    target_dir = './rag_documents'
    print(f"Generating {len(corpus_data)} PDF documents in '{target_dir}'...")
    for idx, doc in enumerate(corpus_data, 1):
        filename = os.path.join(target_dir, doc['filename'])
        generate_pdf(
            filename=filename,
            title=doc['title'],
            category=doc['category'],
            intro=doc['intro'],
            sections=doc['sections'],
            table_data=doc['table_data'],
            bullets=doc['bullets']
        )
        print(f"  [{idx}/{len(corpus_data)}] Generated: {doc['filename']}")

def generate_knowledge_base_pdfs(target_dir: str):
    """
    Generates 38 comprehensive domain PDF manuals covering Freight, Shipping, Customs, Logistics, etc.
    """
    os.makedirs(target_dir, exist_ok=True)
    print(f"Generating {len(corpus_data)} PDF documents in '{target_dir}'...")
    for idx, doc in enumerate(corpus_data, 1):
        filename = os.path.join(target_dir, doc['filename'])
        generate_pdf(
            filename=filename,
            title=doc['title'],
            category=doc['category'],
            intro=doc['intro'],
            sections=doc['sections'],
            table_data=doc['table_data'],
            bullets=doc['bullets']
        )
    print("All documents generated successfully.")

def get_vector_store(emb=None, rag_dir=None, faiss_dir=None):
    """
    Returns the singleton FAISS vector store. Initializes and indexes documents if index does not exist.
    """
    global _vectorstore
    if _vectorstore is not None:
        return _vectorstore

    if emb is None:
        emb = get_embedding_model()

    if rag_dir is None:
        rag_dir = getattr(config, "RAG_DIR", "./rag_documents")
    if faiss_dir is None:
        faiss_dir = getattr(config, "FAISS_DIR", "./faiss_index")

    os.makedirs(rag_dir, exist_ok=True)
    os.makedirs(faiss_dir, exist_ok=True)

    faiss_file = os.path.join(faiss_dir, "index.faiss")

    if FAISS is not None and os.path.exists(faiss_file):
        try:
            _vectorstore = FAISS.load_local(faiss_dir, emb, allow_dangerous_deserialization=True)
            return _vectorstore
        except Exception:
            pass

    # Generate PDFs if directory has fewer than 35 PDFs
    pdf_files = glob.glob(os.path.join(rag_dir, "*.pdf"))
    if len(pdf_files) < 35:
        generate_knowledge_base_pdfs(rag_dir)
        pdf_files = glob.glob(os.path.join(rag_dir, "*.pdf"))

    raw_docs = []
    for pdf_path in pdf_files:
        try:
            doc = fitz.open(pdf_path)
            fname = os.path.basename(pdf_path)

            cat = "Logistics"
            for kw in ["Freight", "Incoterms", "Customs", "Port", "Risk", "Warehousing", "Carrier", "Import", "Transportation", "Compliance", "Inventory", "Delay", "Costs", "Corridors", "Profiles", "Guides"]:
                if kw.lower() in fname.lower():
                    cat = kw
                    break

            for i in range(len(doc)):
                t = doc[i].get_text("text").strip()
                if t and Document is not None:
                    raw_docs.append(Document(
                        page_content=t,
                        metadata={
                            "source": fname,
                            "filepath": pdf_path,
                            "page": i + 1,
                            "total_pages": len(doc),
                            "category": cat
                        }
                    ))
            doc.close()
        except Exception as e:
            print(f"Error parsing {pdf_path}: {e}")
            pass

    if RecursiveCharacterTextSplitter is not None:
        splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
        chunks = splitter.split_documents(raw_docs)
    else:
        chunks = raw_docs

    if FAISS is not None:
        _vectorstore = FAISS.from_documents(chunks, emb)
        _vectorstore.save_local(faiss_dir)
    return _vectorstore


### Step 5: Execute Ingestion & PDF Generation
If no PDFs were uploaded, this step runs `vector_store.py` to generate the 35 baseline PDF shipping manuals.

In [ ]:
import glob
import sys
sys.path.append('/content/freightquote_m4')

pdf_count = len(glob.glob(os.path.join(RAG_DIR, "*.pdf")))
if pdf_count == 0:
    print("No PDFs found in RAG folder. Seeding 35 default shipping manuals...")
    import vector_store
    vector_store.generate_knowledge_base_pdfs(RAG_DIR)
    pdf_count = len(glob.glob(os.path.join(RAG_DIR, "*.pdf")))
    print(f"Successfully generated {pdf_count} baseline PDFs!")
else:
    print(f"Found {pdf_count} PDFs. Proceeding with user documents.")

### Step 6: Text Extraction & Character Chunking
Extracts raw text page-by-page from PDFs and splits them into clean chunks.

In [ ]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_files = glob.glob(os.path.join(RAG_DIR, "*.pdf"))
raw_docs = []

for pdf_path in pdf_files:
    try:
        doc = fitz.open(pdf_path)
        fname = os.path.basename(pdf_path)
        cat = "Logistics"
        for kw in ["Freight", "Incoterms", "Customs", "Port", "Risk", "Warehousing", "Carrier", "Import", "Transportation", "Compliance"]:
            if kw.lower() in fname.lower():
                cat = kw
                break
        
        for i in range(len(doc)):
            t = doc[i].get_text("text").strip()
            if t:
                raw_docs.append(Document(
                    page_content=t,
                    metadata={
                        "source": fname,
                        "filepath": pdf_path,
                        "page": i + 1,
                        "total_pages": len(doc),
                        "category": cat
                    }
                ))
        doc.close()
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
chunks = splitter.split_documents(raw_docs)
print(f"Ingested {len(pdf_files)} files | Extracted {len(raw_docs)} pages | Segmented {len(chunks)} chunks.")

### Step 7: Build Dense FAISS Index
Uses the standard `sentence-transformers/all-MiniLM-L6-v2` embeddings model to generate vectors and saves index files.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading HuggingFace miniLM embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Generating dense vectors and saving FAISS database...")
vs = FAISS.from_documents(chunks, embeddings)
vs.save_local(FAISS_DIR)
print(f"Dense FAISS vector index exported to: {FAISS_DIR}")

### Step 8: Build Sparse BM25 Index
Compiles the sparse BM25 lexical retriever, caching the index in `faiss_index/bm25_index.pkl`.

In [ ]:
import pickle
from rank_bm25 import BM25Okapi

print("Building sparse lexical index...")
tokenized_corpus = [doc.page_content.lower().split() for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)
bm25_path = os.path.join(FAISS_DIR, "bm25_index.pkl")

with open(bm25_path, "wb") as f:
    pickle.dump({"bm25": bm25, "docs": chunks}, f)
print(f"Sparse BM25 index cached to: {bm25_path}")

### Step 9: Establish Absolute Hybrid Retrieval
Defines the hybrid absolute combination module (0.6 dense + 0.4 sparse).

In [ ]:
import numpy as np

def retrieve_hybrid_test(query, k=3):
    dense_res = vs.similarity_search_with_score(query, k=k*2)
    dense_sims = {}
    for doc, dist in dense_res:
        if dist > 1.3: continue
        dense_sims[doc.page_content] = (doc, 1.0 / (1.0 + float(dist)))
        
    STOPWORDS = {"who", "was", "the", "first", "of", "is", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "with", "by"}
    tokens = [t for t in query.lower().split() if t not in STOPWORDS and t.isalnum()]
    if tokens:
        bm25_scores = bm25.get_scores(tokens)
        top_indices = np.argsort(bm25_scores)[::-1][:k*2]
        sparse_res = [(chunks[idx], bm25_scores[idx]) for idx in top_indices if bm25_scores[idx] > 0.5]
    else:
        sparse_res = []
        
    combined = {}
    for content, (doc, sim) in dense_sims.items():
        combined[content] = {"doc": doc, "score": 0.6 * sim, "dense": sim, "sparse": 0.0}
        
    for doc, score in sparse_res:
        content = doc.page_content
        soft_sparse = score / (score + 10.0)
        if content in combined:
            combined[content]["score"] += 0.4 * soft_sparse
            combined[content]["sparse"] = soft_sparse
        else:
            combined[content] = {"doc": doc, "score": 0.4 * soft_sparse, "dense": 0.0, "sparse": soft_sparse}
            
    ranked = sorted(combined.values(), key=lambda x: x["score"], reverse=True)[:k]
    return ranked

### Step 10: Validation Retrieval Test
Runs validation search query and prints alignment metrics.

In [ ]:
q = "What is BAF?"
print(f"🔎 Running test query: '{q}'")
hits = retrieve_hybrid_test(q, k=2)
for idx, hit in enumerate(hits, 1):
    doc = hit["doc"]
    print(f"Hit #{idx}: {doc.metadata['source']} (Page {doc.metadata['page']})")
    print(f"  Hybrid Score: {hit['score']:.4f} [FAISS Dense: {hit['dense']:.3f} | BM25 Sparse: {hit['sparse']:.3f}]")
    print(f"  Snippet: \"{doc.page_content[:150]}...\"")

### Step 11: Build Status Summary
Prints final index metadata and verification diagnostics.

In [ ]:
import datetime
print("==========================================================")
print("🎉 RAG BUILD PROCESS COMPLETED SUCCESSFULLY!")
print("==========================================================")
print(f"Index Location: {FAISS_DIR}")
print(f"Documents:      {len(pdf_files)}")
print(f"Chunks:         {len(chunks)}")
print(f"FAISS Index:    {'Verified' if os.path.exists(os.path.join(FAISS_DIR, 'index.faiss')) else 'Missing'}")
print(f"BM25 Index:     {'Verified' if os.path.exists(os.path.join(FAISS_DIR, 'bm25_index.pkl')) else 'Missing'}")
print(f"Timestamp:      {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("==========================================================")